In [0]:
from pyspark.sql.functions import col
df_gold_1 = spark.table("crypto.crypto_schema.silver_crypto")

df_gold_1 = df_gold_1.filter(col("market_cap_rank")<=10).select(col("name"),col("current_price"),col("market_cap"),col("market_cap_rank"))\
                     .orderBy(col("market_cap_rank").asc())

df_gold_1.display()

name,current_price,market_cap,market_cap_rank
Bitcoin,71526.0,1430664101603,1
Ethereum,2110.61,254719092775,2
Tether,0.999923,183977928445,3
BNB,660.99,90135762008,4
XRP,1.42,87094294417,5
USDC,0.999953,78879881990,6
Solana,89.03,50848755192,7
TRON,0.289752,27453027555,8
Figure Heloc,1.008,15924422768,9
Dogecoin,0.096653,14830127580,10


In [0]:
df_gold_1.write.format('delta').mode('overwrite')\
          .saveAsTable("crypto.crypto_schema.gold_top10_market_cap")

In [0]:
df_gold_2 = spark.table("crypto.crypto_schema.silver_crypto")

df_gold_2 = df_gold_2.select(col('name'),col('current_price'),col('total_volume')).orderBy(col('total_volume').desc()).limit(10)

df_gold_2.display()

df_gold_2.write.format('delta').mode('overwrite')\
         .saveAsTable("crypto.crypto_schema.gold_top10_volume")

name,current_price,total_volume
Tether,0.999923,7.7377789186E10
Bitcoin,71526.0,4.735715175E10
Ethereum,2110.61,2.9333504285E10
USDC,0.999953,1.2827626646E10
Solana,89.03,4.560141456E9
XRP,1.42,2.510929902E9
USD1,0.999221,1.489398226E9
Dogecoin,0.096653,1.340430182E9
Figure Heloc,1.008,1.014023556E9
BNB,660.99,9.67627015E8


In [0]:
from pyspark.sql.functions import sum as spark_sum,round
from pyspark.sql.window import Window as w

df_gold_3 = spark.table("crypto.crypto_schema.silver_crypto")

window_spec = w.rowsBetween(w.unboundedPreceding,w.unboundedFollowing)
total_market_cap = spark_sum('market_cap').over(window_spec)

df_gold_3 = df_gold_3.withColumn('total_market_cap',total_market_cap)

df_gold_3 = df_gold_3.withColumn("dominance_%",round((col('market_cap')/col('total_market_cap')*100),2))

df_gold_3 = df_gold_3.select(col('name'),col('market_cap'),col('dominance_%')).orderBy(col('dominance_%').desc())

df_gold_3.display()

df_gold_3.write.format('delta').mode('overwrite')\
               .saveAsTable('crypto.crypto_schema.gold_market_cap_dominance')

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


name,market_cap,dominance_%
Bitcoin,1430664101603,58.25
Ethereum,254719092775,10.37
Tether,183977928445,7.49
BNB,90135762008,3.67
XRP,87094294417,3.55
USDC,78879881990,3.21
Solana,50848755192,2.07
TRON,27453027555,1.12
Figure Heloc,15924422768,0.65
Dogecoin,14830127580,0.6
